# Text Classification with RNN, LSTM and GRU

This notebook builds a complete mental model for sequence classification. It follows text from **raw sentences** to **token IDs**, then makes unequal-length sequences usable with **padding**, turns IDs into learned vectors with an **Embedding** layer, and compares **SimpleRNN**, **LSTM**, and **GRU** classifiers.

**Learning goals**

1. Explain how raw text becomes numeric token IDs.
2. Understand why records have different sequence lengths and how padding solves batching.
3. Interpret vocabulary size, embedding dimension, and sequence length.
4. Train and evaluate recurrent neural network models for binary sentiment/spam classification.

## 1. The complete pipeline

`raw text -> clean/split text -> vocabulary -> token IDs -> pad/truncate -> embedding vectors -> RNN/LSTM/GRU -> sigmoid probability -> class`

A neural network cannot read words directly. It receives numbers. Crucially, a token ID is only a **lookup address**, not a numerical meaning: ID 19 is not inherently closer to ID 20 than to ID 900. The Embedding layer learns the meaningful vector representation.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)

## 2. Why padding is needed

Every review/message contains a different number of tokens. For example, `['great', 'movie']` has length 2 while a long review can have hundreds of tokens. A batch must be a rectangular tensor, so we choose a fixed `max_len`.

- **Padding** adds the special value `0` to shorter sequences.
- **Truncation** removes tokens from sequences longer than `max_len`.
- `padding='post'` appends zeros; `padding='pre'` adds zeros at the start.
- `truncating='post'` keeps the beginning; `truncating='pre'` keeps the end.

When using zero padding, set `mask_zero=True` in Embedding so the recurrent layer can ignore padded positions.

In [ ]:
toy_sequences = [[10, 25, 73, 42], [9, 8], [1, 2, 3, 4, 5, 6, 7]]
print('Original lengths:', [len(s) for s in toy_sequences])
print(pad_sequences(toy_sequences, maxlen=5, padding='post', truncating='post'))

## 3. Path A: IMDB data already comes as token IDs

The Keras IMDB dataset is preprocessed: each review is already an integer sequence. `num_words=10000` keeps the 10,000 most frequent tokens. In this dataset, index offsets are reserved for special tokens; inspect the word index before decoding. This is different from the SMS section below, where we build token IDs from raw strings.

In [ ]:
from tensorflow.keras.datasets import imdb

VOCAB_SIZE = 10_000
MAX_LEN = 200
(x_train_raw, y_train), (x_test_raw, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print('Training reviews:', len(x_train_raw))
print('Example token IDs:', x_train_raw[0][:20])
print('Different lengths:', [len(x_train_raw[i]) for i in range(3)])
print('Min / max train length:', min(map(len, x_train_raw)), max(map(len, x_train_raw)))

In [ ]:
# Convert ragged Python lists into fixed-size tensors.
x_train = pad_sequences(x_train_raw, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = pad_sequences(x_test_raw, maxlen=MAX_LEN, padding='post', truncating='post')
print('Padded train shape:', x_train.shape)  # (reviews, 200)
print('Labels:', y_train[:10])               # 0 = negative, 1 = positive

## 4. Embedding: IDs become dense learned vectors

`Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)` is a trainable table with shape `(VOCAB_SIZE, EMBED_DIM)`. For each token ID in a review, it returns one row from that table.

With 10,000 vocabulary entries and 128 dimensions, the embedding table has `10,000 x 128 = 1,280,000` trainable values. The output for a batch has shape `(batch_size, MAX_LEN, EMBED_DIM)`. This is often described as an **embedding lookup**; it is not an outer product.

In [ ]:
EMBED_DIM = 128
embedding = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, mask_zero=True)
embedded_batch = embedding(x_train[:4])
print('Embedding output shape:', embedded_batch.shape)  # (4, 200, 128)

## 5. Recurrent layers: what changes between RNN, LSTM and GRU

All three read the embedding sequence one time step at a time and produce a summary used for classification.

| Layer | Main idea | Strength | Typical limitation |
|---|---|---|---|
| SimpleRNN | One hidden state | Small, good for learning the basics | Vanishing/exploding gradients on long dependencies |
| LSTM | Cell state plus input, forget, output gates | Strong long-term memory | More parameters and slower |
| GRU | Update and reset gates | Often close to LSTM with fewer parameters | Slightly less flexible than LSTM |

For binary classification, the final `Dense(1, activation='sigmoid')` gives a probability between 0 and 1. Binary cross-entropy compares that probability against the true label.

In [ ]:
def build_model(kind='lstm'):
    recurrent = {
        'rnn': SimpleRNN(128),
        'lstm': LSTM(128),
        'gru': GRU(128),
    }[kind]
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True),
        recurrent,
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model('lstm')  # Change to 'rnn' or 'gru' to compare.
model.summary()

In [ ]:
# Start with a small epoch count while learning; increase after checking validation behavior.
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=3,
    batch_size=64,
)
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f} | Test accuracy: {accuracy:.4f}')

## 6. Path B: raw SMS text to token IDs

Here the dataset contains actual message strings. `Tokenizer.fit_on_texts(X_train)` learns a vocabulary from the training split only - this prevents information from the test set leaking into training. `texts_to_sequences` then maps each word to its ID.

`oov_token='<OOV>'` gives unseen words a known fallback ID. The tokenizer normally lowercases and removes configured punctuation, then splits text into tokens. Different tokenizers can use subwords or more advanced rules, but the core idea is the same: text becomes a sequence of integer IDs.

In [ ]:
import io, zipfile, requests, pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
response = requests.get(url, timeout=30)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open('SMSSpamCollection') as f:
        sms = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

sms['label'] = sms['label'].map({'ham': 0, 'spam': 1})
X_train_text, X_test_text, y_train_sms, y_test_sms = train_test_split(
    sms['message'], sms['label'], test_size=0.2, random_state=SEED, stratify=sms['label']
)
print(sms.shape)
print(sms['label'].value_counts())

In [ ]:
SMS_VOCAB = 10_000
SMS_MAX_LEN = 50
tokenizer = Tokenizer(num_words=SMS_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

train_sequences = tokenizer.texts_to_sequences(X_train_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)
print('Raw message:', X_train_text.iloc[0])
print('Token IDs:  ', train_sequences[0])
print('Vocabulary learned:', len(tokenizer.word_index))

X_train_sms = pad_sequences(train_sequences, maxlen=SMS_MAX_LEN, padding='post', truncating='post')
X_test_sms = pad_sequences(test_sequences, maxlen=SMS_MAX_LEN, padding='post', truncating='post')
print('Padded shape:', X_train_sms.shape)

In [ ]:
def build_sms_model(kind='gru'):
    recurrent = {'rnn': SimpleRNN(64), 'lstm': LSTM(64), 'gru': GRU(64)}[kind]
    model = Sequential([
        Embedding(SMS_VOCAB, 128, mask_zero=True),
        recurrent,
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

sms_model = build_sms_model('gru')
sms_model.summary()
# sms_model.fit(X_train_sms, y_train_sms, validation_split=0.2, epochs=5, batch_size=32)
# sms_model.evaluate(X_test_sms, y_test_sms)

## 7. Practical checklist and common mistakes

- Fit the tokenizer on **training text only**.
- Use the same tokenizer and the same padding settings for train, validation, and test data.
- Reserve ID `0` for padding when using `mask_zero=True`; never let a real word use it.
- `input_dim` must cover the largest possible token ID. With `num_words=10000`, use at least 10,000.
- `MAX_LEN` is a design choice: inspect length percentiles and balance memory, speed, and lost text.
- Compare validation metrics, not only training accuracy; a large gap can indicate overfitting.
- Start with GRU/LSTM for longer text dependencies; SimpleRNN is mainly a useful baseline.

### Key takeaways

A token ID is an index. Padding makes variable-length records batchable. Embeddings learn dense word features. RNN, LSTM, and GRU learn from order, with gated models usually handling longer context more reliably.